In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from PIL import Image

import ipywidgets as widgets
from IPython.display import display, clear_output
# --------- 設定 ----------
csv_path = "../tests/outputs/grid/vortex/depth4/time_vs_total_T.csv"
img_dir = "../tests/outputs/grid/vortex/depth4/"
img_ext = ".png"
img_pattern = "T.{f:04d}"
# ------------------------

df = pl.read_csv(csv_path)

times = df["time"].to_numpy()
total_T = df["total_T"].to_numpy()
total_T_normalized = total_T / total_T[0]

def load_image_by_frame(f: int):
    path = pathlib.Path(img_dir) / (img_pattern.format(f=f) + img_ext)
    return np.array(Image.open(path))

plt.ioff()  # Jupyterでの再描画チラつき抑制
fig, (ax_plot, ax_img) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios":[1.2, 1]})

ax_plot.plot(times, total_T_normalized, lw=1, marker="o", markersize=4)
(marker,) = ax_plot.plot([times[0]], [total_T_normalized[0]], marker="o", markersize=8, linestyle="None", color="red")
ax_plot.set_xlabel("time")
ax_plot.set_ylabel("normalized total T")
ax_plot.set_title("normalized total T vs time")
ax_plot.set_ylim(0.995, 1.008)

im = ax_img.imshow(load_image_by_frame(0))
ax_img.set_axis_off()
ax_img.set_title(f"time={times[0]:.3f}")

out = widgets.Output()

def update(idx: int):
    idx = int(idx)

    # marker
    marker.set_data([times[idx]], [total_T_normalized[idx]])

    # image
    im.set_data(load_image_by_frame(idx))
    ax_img.set_title(f"time={times[idx]:.3f}")

    # 表示更新（Outputに描画し直す）
    with out:
        clear_output(wait=True)
        display(fig)

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(times)-1,
    step=1,
    description="index",
    continuous_update=False,  # 重い画像なら False 推奨
)

# indexに対応するframe番号も表示
label = widgets.HTML()
def sync_label(change=None):
    i = int(slider.value)
    label.value = f"<b>time</b>: {times[i]:.3f} &nbsp;&nbsp; <b>normalized total T</b>: {total_T_normalized[i]:.6g}"

def on_slider_change(change):
    if change["name"] == "value":
        sync_label()
        update(change["new"])

slider.observe(on_slider_change, names="value")

# 初期表示
sync_label()
update(0)
display(widgets.VBox([widgets.HBox([slider, label]), out]))


In [ ]:
# GIF保存用の関数
import imageio
from io import BytesIO

def save_animation_as_gif(output_path: str, duration: float = 0.2):
    """
    AnimationをGIFとして保存
    
    Parameters
    ----------
    output_path : str
        出力GIFファイルのパス
    duration : float
        各フレームの表示時間（秒）
    """
    # 一時的にfigureを再作成
    fig_gif, (ax_plot_gif, ax_img_gif) = plt.subplots(
        1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.2, 1]}
    )
    
    # プロットを設定
    ax_plot_gif.plot(times, total_T_normalized, lw=1, marker="o", markersize=4)
    (marker_gif,) = ax_plot_gif.plot(
        [times[0]], [total_T_normalized[0]], marker="o", markersize=8, linestyle="None", color="red"
    )
    ax_plot_gif.set_xlabel("time")
    ax_plot_gif.set_ylabel("normalized total T")
    ax_plot_gif.set_title("normalized total T vs time (Limited Linear - fine)")
    ax_plot_gif.set_ylim(0.995, 1.008)
    
    im_gif = ax_img_gif.imshow(load_image_by_frame(0))
    ax_img_gif.set_axis_off()
    ax_img_gif.set_title(f"time={times[0]:.3f}")
    
    frames_list = []
    for idx in range(len(times)):
        # markerを更新
        marker_gif.set_data([times[idx]], [total_T_normalized[idx]])
        
        # 画像を更新
        im_gif.set_data(load_image_by_frame(idx))
        ax_img_gif.set_title(f"time={times[idx]:.3f}")
        
        # フレームをバッファに保存
        buf = BytesIO()
        fig_gif.savefig(buf, format="png", bbox_inches="tight", dpi=100)
        buf.seek(0)
        frame = Image.open(buf)
        frames_list.append(np.array(frame))
        buf.close()
    
    plt.close(fig_gif)
    
    # GIFとして保存
    imageio.mimsave(output_path, frames_list, duration=duration, loop=0)
    print(f"GIF saved to: {output_path}")

# 使用例（コメントアウト）:
save_animation_as_gif("animation.gif", duration=0.2)